# 02 - Pengunduhan & Pengorganisasian Dataset DeepFashion In-Shop
### Proyek CITRA SERUPA — Sistem Rekomendasi Gambar Fashion Berdasarkan Kemiripan Visual

---

**Tujuan Notebook:**
1. Menghubungkan runtime Google Colab ke Google Drive (`CITRA_SERUPA/dataset/`).
2. Mengunduh dataset resmi DeepFashion (In-shop Clothes Retrieval) melalui mirror publik Hugging Face (`Marqo/deepfashion-inshop`).
3. Mengurasi **10.000 gambar fashion berpasangan** (berdasarkan `item_id` unik) agar setiap produk memiliki foto tampak depan, samping, atau belakang untuk evaluasi *visual similarity*.
4. Menyimpan 10.000 gambar ke Google Drive dan menyusun file indeks `metadata.csv`.

## 1. Hubungkan Google Colab ke Google Drive & Siapkan Direktori

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Tentukan direktori penyimpanan proyek
drive_root = '/content/drive/MyDrive/CITRA_SERUPA'
if not os.path.exists(drive_root):
    drive_root = '/content/drive/MyDrive/CITRA-SERUPA'

dataset_dir = os.path.join(drive_root, 'dataset')
images_dir = os.path.join(dataset_dir, 'images')

os.makedirs(images_dir, exist_ok=True)
print(f"✅ Direktori dataset siap di: {dataset_dir}")
print(f"📁 Folder gambar: {images_dir}")

## 2. Instalasi Pustaka Datasets (Hugging Face)

In [ ]:
!pip install -q datasets

## 3. Kurasi & Simpan 10.000 Gambar Berpasangan ke Google Drive

In [ ]:
import re
import os
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from PIL import Image

print("⏳ Memuat dataset DeepFashion In-shop...")
dataset = load_dataset("Marqo/deepfashion-inshop", split="data")
print(f"✅ Total data tersedia: {len(dataset):,} gambar.")

TARGET_IMAGES = 10000
print(f"🎯 Target kurasi: {TARGET_IMAGES:,} gambar berpasangan...")

# Fungsi ekstraksi ID item pakaian unik
def extract_item_code(item_id_str):
    match = re.match(r"(.*_id_\d+)", str(item_id_str))
    if match:
        return match.group(1)
    parts = str(item_id_str).split("_")
    return "_".join(parts[:4]) if len(parts) >= 4 else str(item_id_str)

# Kelompokkan index data per item_code
print("🔍 Mengelompokkan gambar berdasarkan ID produk unik...")
items_map = {}
for idx, row in enumerate(dataset):
    raw_id = row['item_ID']
    code = extract_item_code(raw_id)
    if code not in items_map:
        items_map[code] = []
    items_map[code].append(idx)

# Prioritaskan pakaian yang memiliki minimal 2 foto sudut berbeda
curated_indices = []
selected_items_count = 0

for code, indices in items_map.items():
    if len(indices) >= 2:
        curated_indices.extend(indices)
        selected_items_count += 1
        if len(curated_indices) >= TARGET_IMAGES:
            curated_indices = curated_indices[:TARGET_IMAGES]
            break

print(f"✅ Terpilih {len(curated_indices):,} gambar dari {selected_items_count:,} item pakaian berbeda.")

# Simpan gambar ke Google Drive dan catat metadata
print("\n💾 Menyimpan 10.000 gambar ke Google Drive...")
metadata_records = []

for rank, idx in enumerate(tqdm(curated_indices, desc="Menyimpan Gambar ke Drive")):
    row = dataset[idx]
    raw_id = row['item_ID']
    item_code = extract_item_code(raw_id)
    category = row.get('category2') or row.get('category1') or 'unknown'
    color = row.get('color') or 'unknown'

    # Standardisasi nama file
    clean_suffix = raw_id.split('_')[-1]
    file_name = f"{item_code}_{clean_suffix}.jpg"
    file_name = "".join(c for c in file_name if c.isalnum() or c in ('_', '.', '-'))
    save_path = os.path.join(images_dir, file_name)

    # Simpan file gambar JPEG
    img = row['image']
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.save(save_path, 'JPEG', quality=90)

    metadata_records.append({
        'image_name': file_name,
        'item_id': item_code,
        'full_id': raw_id,
        'category': category,
        'color': color,
        'path': os.path.join('images', file_name)
    })

# Simpan tabel metadata ke file CSV
df_meta = pd.DataFrame(metadata_records)
csv_path = os.path.join(dataset_dir, 'metadata.csv')
df_meta.to_csv(csv_path, index=False)

print("\n🎉 Pengunduhan dan pengorganisasian 10.000 gambar SELESAI!")
print(f"📁 Folder Gambar: {images_dir}")
print(f"📄 File Metadata: {csv_path}")
print(f"📊 Total data tersimpan: {len(df_meta):,} gambar.")

## 4. Verifikasi Data & Menampilkan Sampel Pasangan Gambar Serupa

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# 1. Tampilkan cuplikan tabel metadata
df = pd.read_csv(csv_path)
print("📋 5 Data Teratas Metadata:")
display(df.head())

# 2. Distribusi Kategori Busana
print("\n📊 Distribusi Kategori Pakaian:")
print(df['category'].value_counts())

# 3. Visualisasikan contoh pasangan foto untuk 1 item pakaian (Ground Truth Kemiripan)
sample_item = df['item_id'].value_counts().index[0]
sample_images = df[df['item_id'] == sample_item]['image_name'].tolist()

print(f"\n🖼️ Variasi foto untuk item baju yang sama ({sample_item}):")
n_photos = min(len(sample_images), 4)
fig, axes = plt.subplots(1, n_photos, figsize=(3 * n_photos, 4))
if n_photos == 1:
    axes = [axes]
for i in range(n_photos):
    img_path = os.path.join(images_dir, sample_images[i])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(sample_images[i].split('_')[-1])
    axes[i].axis('off')
plt.tight_layout()
plt.show()